# Original whole-heart phase playback

This example reads ten **stored CONRAD phase consensuses** sequentially using
PyVista from this branch. The required `examples.download_heart_motion()` implementation
is included in `pyvista/examples/heart.py`; no separately built wheel or local heart
assets are required.

From the root of a clone of branch `v0.0.1`, use Python 3.10 or newer:

```sh
python3 -m venv .venv-heart
.venv-heart/bin/python -m pip install -e '.[io]' jupyterlab ipykernel psutil
.venv-heart/bin/python -m ipykernel install --user --name pyvista-heart --display-name 'PyVista heart (v0.0.1)'
.venv-heart/bin/jupyter lab heart_animation.ipynb
```

Select **PyVista heart (v0.0.1)** and **Run All**. Installing from this clone is
required because released PyVista does not contain the downloader. An OpenGL renderer
is required. First use downloads 78,152,158 bytes from the provider; later runs verify
and reuse the cache. Set `PYVISTA_USERDATA_PATH` before starting Jupyter to choose the
cache directory. The notebook writes `heart_preview.gif` in its working directory and
displays it inline.

**Source:** Unberath et al., *Open-source 4D Statistical Shape Model of the Heart
for X-ray Projection Imaging*, ISBI 2015, 739–742.
[Provider and original data](https://www5.cs.fau.de/conrad/data/heart-model/index.html).
Archive SHA-256: `4c68caa46eb96ee83f604011b2ace79c2e30297220f7e0cb2ae00b0bd7b03b87`.
The provider gives no explicit standalone dataset license. CONRAD's GPL code
license is **not** asserted to license the separate data archive.

All four cavities, LV myocardium, and aorta share each source phase. Missing are
pulmonary artery, detailed valves, coronaries, pericardium, and separate RV/atrial
myocardial walls. These are statistical shapes derived from registered CT, not a
patient recording or the provider's full two-level spline phantom. Coordinates
are unchanged; millimeters are inferred from CT/model scale, not a file unit tag.
Source phases are 0.0–0.9 of a cycle, with no measured duration. The **0.5 s display
period** is a chosen setting, not an acquisition measurement. No interpolation,
deformation, smoothing, decimation, or physiological correction is applied.


In [ ]:
from __future__ import annotations

import gc
import hashlib
from importlib.metadata import version
import inspect
import platform

from IPython.display import Image
import numpy as np
import psutil

import pyvista as pv
from pyvista import examples

pv.OFF_SCREEN = True
versions = {
    name: version(name)
    for name in ('pyvista', 'vtk', 'numpy', 'pyvista-zstd', 'imageio', 'pillow', 'nbclient')
}
print('Python:', platform.python_version())
print('Installed PyVista:', inspect.getfile(pv))
print(versions)

filenames = examples.download_heart_motion()
assert len(filenames) == 10
print('First cached frame:', filenames[0])

The actor and mapper are reused. Every iteration reads one complete source
phase and replaces the mapper dataset; geometry is never interpolated or edited
in place. Fixed connectivity is checked without treating it as proof of material
point identity. Source registration documents anatomical correspondence.

Five serial cycles check memory retention; only the first is encoded. Geometry
buffering is bounded to the current and replacement frame. The GIF encoder can
retain ten small 400 × 320 raster images. RSS measurements include native CPU
allocations, not GPU memory. Separate validation compares every displayed phase
with an independent reading of the original files; it does not establish cardiac
mechanics or physiological validity.


In [ ]:
mesh = pv.read(filenames[0])
topology = mesh.faces.copy()
parts = mesh.field_data['part_names'].tolist()
colors = ['#bb3754', '#4b85c2', '#e6a64c', '#74bdcd', '#e8d6b0', '#df6461']
lut = pv.LookupTable(cmap=colors, n_values=6, scalar_range=(-0.5, 5.5))
rgba = lut.values.copy()
rgba[0, 3] = 70  # Reveal the cavities through the LV myocardium.
lut.values = rgba

pl = pv.Plotter(notebook=False, window_size=(400, 320))
cycle_rss = []
displayed_frames = []
display_cycle_seconds = 0.5
try:
    actor = pl.add_mesh(
        mesh, scalars='part', preference='point', cmap=lut, clim=(-0.5, 5.5), show_scalar_bar=False
    )
    mapper = actor.mapper
    pl.set_background('#101b2c')
    pl.camera_position = [(40, -180, 180), (0, 0, 0), (-1, 0, 0)]
    pl.enable_parallel_projection()
    pl.camera.zoom(0.85)
    pl.add_legend(list(zip(parts, colors, strict=True)), size=(0.38, 0.30), bcolor='#101b2c')
    pl.add_text(
        'CONRAD phase consensuses | mm inferred', position='upper_left', font_size=8, color='white'
    )
    pl.show(interactive=False, auto_close=False)
    pl.open_gif('heart_preview.gif', fps=len(filenames) / display_cycle_seconds)
    for cycle in range(5):
        phases = []
        for index, filename in enumerate(filenames):
            mesh = pv.read(filename)
            phase = mesh.field_data['cardiac_phase'].item()
            assert phase == index / 10
            assert np.array_equal(mesh.faces, topology)
            assert mesh.n_points == 16430
            assert mesh.n_cells == 32836
            phases.append(phase)
            mapper.dataset = mesh
            assert actor.mapper is mapper
            pl.add_text(
                f'Cardiac phase {phase:.1f}',
                name='phase',
                font_size=10,
                position='lower_left',
                color='white',
            )
            if cycle == 0:
                pl.write_frame()
                displayed_frames.append(
                    (phase, hashlib.sha256(mapper.dataset.points.tobytes()).hexdigest())
                )
            else:
                pl.render()
        gc.collect()
        cycle_rss.append(psutil.Process().memory_info().rss)
finally:
    pl.close()

print('Source phases:', phases)
print('Displayed phase digests:', displayed_frames)
print('Cycle-end RSS (bytes):', cycle_rss)
print('Cycle-end RSS (MiB):', [round(value / 1024**2, 2) for value in cycle_rss])
print('Cycle-end growth (MiB):', round((cycle_rss[-1] - cycle_rss[0]) / 1024**2, 2))
Image(filename='heart_preview.gif')